# Greedy Forward Selection - PLS
Tự động chọn N điểm V tốt nhất từ file CSV đã xử lý.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import cross_val_predict, LeaveOneOut
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')
print('✅ Libraries loaded')

---
## ⚙️ CONFIG

In [ ]:
# ================================================================
#  File CSV đã xử lý (từ PLS_custom_V hoặc tự tạo)
#  Yêu cầu: cột đầu = target (y), các cột còn lại = features (X)
# ================================================================

FILE_PATH   = 'CUCOMOF_processed.csv'   # <-- đổi tên file
TARGET_COL  = 'glucose_mM'              # <-- tên cột y

# Số features muốn chọn (None = chạy hết rồi tự tìm điểm dừng)
N_SELECT    = 5                         # <-- đổi số lượng

# ================================================================
print(f'File: {FILE_PATH}')
print(f'Target: {TARGET_COL}')
print(f'N select: {N_SELECT}')

---
## 1. Load Data

In [ ]:
df = pd.read_csv(FILE_PATH)
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print()
display(df)

y            = df[TARGET_COL].values
feature_cols = [c for c in df.columns if c != TARGET_COL]
X_all        = df[feature_cols].values   # (n_samples, n_features)

print(f'\nSamples  : {X_all.shape[0]}')
print(f'Features : {X_all.shape[1]}  →  {feature_cols}')
print(f'y range  : {y.min()} – {y.max()}')

---
## 2. Greedy Forward Selection

In [ ]:
def eval_pls_loo(X, y):
    """PLS LOO-CV, tự chọn n_components tốt nhất."""
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)
    best_r2, best_rmse, best_nc = -999, 999, 1
    for nc in range(1, min(X.shape[1], len(y)-1) + 1):
        pls  = PLSRegression(n_components=nc)
        yp   = cross_val_predict(pls, Xs, y, cv=LeaveOneOut()).ravel()
        r2   = r2_score(y, yp)
        rmse = np.sqrt(mean_squared_error(y, yp))
        if r2 > best_r2:
            best_r2, best_rmse, best_nc = r2, rmse, nc
    return best_r2, best_rmse, best_nc

# Số bước tối đa
max_steps    = N_SELECT if N_SELECT else len(feature_cols)
max_steps    = min(max_steps, len(feature_cols), len(y) - 1)

remaining    = list(range(len(feature_cols)))   # index chưa chọn
selected_idx = []                               # index đã chọn
history      = []                               # lịch sử mỗi bước

print(f'Greedy Forward Selection  (max {max_steps} steps)\n')
print(f'{"Step":<6} {"Added Feature":<22} {"R2_LOO":<10} {"RMSE_LOO":<12} {"nComp":<7} {"ΔR2"}')
print('-' * 70)

prev_r2 = 0
for step in range(max_steps):
    best_r2, best_rmse, best_nc = -999, 999, 1
    best_feat_idx = None

    for fi in remaining:
        trial_idx = selected_idx + [fi]
        X_trial   = X_all[:, trial_idx]
        if np.any(np.isnan(X_trial)):
            continue
        r2, rmse, nc = eval_pls_loo(X_trial, y)
        if r2 > best_r2:
            best_r2, best_rmse, best_nc, best_feat_idx = r2, rmse, nc, fi

    if best_feat_idx is None:
        print('Không còn feature hợp lệ.')
        break

    selected_idx.append(best_feat_idx)
    remaining.remove(best_feat_idx)
    delta = best_r2 - prev_r2
    fname = feature_cols[best_feat_idx]

    history.append({
        'step'       : step + 1,
        'added'      : fname,
        'R2_LOO'     : round(best_r2,  4),
        'RMSE_LOO'   : round(best_rmse, 4),
        'n_components': best_nc,
        'delta_R2'   : round(delta, 4),
        'selected'   : [feature_cols[i] for i in selected_idx]
    })

    print(f'{step+1:<6} {fname:<22} {best_r2:<10.4f} {best_rmse:<12.4f} {best_nc:<7} {delta:+.4f}')
    prev_r2 = best_r2

hist_df  = pd.DataFrame(history)
best_step = hist_df['R2_LOO'].idxmax()
optimal  = history[best_step]

print(f'\n✅ Optimal tại step {optimal["step"]}: {optimal["selected"]}')
print(f'   R²(LOO) = {optimal["R2_LOO"]}   RMSE = {optimal["RMSE_LOO"]} mM')

---
## 3. Visualization Quá trình Selection

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

steps = hist_df['step']

# R² vs step
ax1 = axes[0]
ax1.plot(steps, hist_df['R2_LOO'], 'bo-', linewidth=2, markersize=8)
ax1.axvline(x=optimal['step'], color='red', linestyle='--', lw=1.5, label=f'Optimal=step {optimal["step"]}')
for _, row in hist_df.iterrows():
    ax1.annotate(f"{row['R2_LOO']:.3f}\n{row['added']}",
                 (row['step'], row['R2_LOO']),
                 textcoords='offset points', xytext=(6, 6), fontsize=7.5)
ax1.set_xlabel('Step', fontsize=11)
ax1.set_ylabel('R² (LOO-CV)', fontsize=11)
ax1.set_title('R² vs Step', fontsize=12, fontweight='bold')
ax1.set_xticks(steps)
ax1.legend(); ax1.grid(True, alpha=0.3)

# RMSE vs step
ax2 = axes[1]
ax2.plot(steps, hist_df['RMSE_LOO'], 'rs-', linewidth=2, markersize=8)
ax2.axvline(x=optimal['step'], color='red', linestyle='--', lw=1.5, label=f'Optimal=step {optimal["step"]}')
for _, row in hist_df.iterrows():
    ax2.annotate(f"{row['RMSE_LOO']:.3f}",
                 (row['step'], row['RMSE_LOO']),
                 textcoords='offset points', xytext=(6, 4), fontsize=8)
ax2.set_xlabel('Step', fontsize=11)
ax2.set_ylabel('RMSE (mM)', fontsize=11)
ax2.set_title('RMSE vs Step', fontsize=12, fontweight='bold')
ax2.set_xticks(steps)
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.suptitle('Greedy Forward Selection', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('greedy_selection_curve.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Train PLS với bộ feature tối ưu

In [ ]:
opt_idx  = [feature_cols.index(f) for f in optimal['selected']]
X_opt    = X_all[:, opt_idx]

scaler   = StandardScaler()
X_opt_s  = scaler.fit_transform(X_opt)

pls_opt  = PLSRegression(n_components=optimal['n_components'])
pls_opt.fit(X_opt_s, y)

y_train  = pls_opt.predict(X_opt_s).ravel()
y_loo    = cross_val_predict(pls_opt, X_opt_s, y, cv=LeaveOneOut()).ravel()

r2_train  = r2_score(y, y_train)
r2_loo    = r2_score(y, y_loo)
rmse_train = np.sqrt(mean_squared_error(y, y_train))
rmse_loo   = np.sqrt(mean_squared_error(y, y_loo))

print('=' * 50)
print(f'  PLS Optimal Model  (nComp={optimal["n_components"]})')
print('=' * 50)
print(f'  Features   : {optimal["selected"]}')
print(f'  Train  R²  = {r2_train:.4f}   RMSE = {rmse_train:.4f} mM')
print(f'  LOO-CV R²  = {r2_loo:.4f}   RMSE = {rmse_loo:.4f} mM')
print('=' * 50)

---
## 5. Kết quả Final

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Predicted vs Actual
ax1 = axes[0]
ax1.scatter(y, y_loo, s=100, color='steelblue', zorder=5)
lim = [y.min()-0.4, y.max()+0.4]
ax1.plot(lim, lim, 'r--', lw=1.5, label='Ideal')
for a, p in zip(y, y_loo):
    ax1.annotate(f'{a}', (a, p), textcoords='offset points', xytext=(5,4), fontsize=8)
ax1.set_xlabel('Actual', fontsize=11); ax1.set_ylabel('Predicted', fontsize=11)
ax1.set_title(f'Predicted vs Actual (LOO)\nR²={r2_loo:.4f}  RMSE={rmse_loo:.4f}', fontweight='bold')
ax1.legend(); ax1.grid(True, alpha=0.3)

# Residuals
ax2 = axes[1]
res = y_loo - y
ax2.bar(range(len(y)), res, color=['#e74c3c' if r < 0 else '#3498db' for r in res],
        alpha=0.85, edgecolor='black')
ax2.axhline(0, color='black', lw=1)
ax2.set_xticks(range(len(y)))
ax2.set_xticklabels([str(v) for v in y], fontsize=9)
for i, ri in enumerate(res):
    ax2.annotate(f'{ri:.3f}', (i, ri), textcoords='offset points',
                 xytext=(0, 5 if ri>=0 else -13), ha='center', fontsize=8)
ax2.set_xlabel('Sample', fontsize=11); ax2.set_ylabel('Residual', fontsize=11)
ax2.set_title('Residuals (LOO-CV)', fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

# Feature importance (VIP)
ax3 = axes[2]
def vip_scores(model):
    t = model.x_scores_; w = model.x_weights_; q = model.y_loadings_
    p, h = w.shape
    s = np.diag(t.T @ t @ q.T @ q).reshape(h, -1)
    total_s = np.sum(s)
    return np.array([np.sqrt(p * (s.T @ np.array(
        [(w[i,j]/np.linalg.norm(w[:,j]))**2 for j in range(h)])) / total_s)[0]
        for i in range(p)])

vip = vip_scores(pls_opt)
short_labels = [f.replace('V_','') for f in optimal['selected']]
ax3.bar(short_labels, vip, color=['#e74c3c' if v>=1 else '#95a5a6' for v in vip],
        alpha=0.85, edgecolor='black')
ax3.axhline(1.0, color='red', linestyle='--', lw=1.5, label='VIP=1.0')
ax3.set_xlabel('Feature', fontsize=11); ax3.set_ylabel('VIP Score', fontsize=11)
ax3.set_title('VIP Scores', fontweight='bold')
ax3.tick_params(axis='x', rotation=45)
ax3.legend(); ax3.grid(True, alpha=0.3, axis='y')

plt.suptitle(
    f'PLS Optimal  |  Features: {optimal["selected"]}  nComp={optimal["n_components"]}  '
    f'R²={r2_loo:.4f}  RMSE={rmse_loo:.4f}mM',
    fontsize=10, fontweight='bold'
)
plt.tight_layout()
plt.savefig('greedy_pls_final.png', dpi=150, bbox_inches='tight')
plt.show()

# Bảng kết quả
print('\n=== Prediction Table ===')
df_res = pd.DataFrame({
    'Actual'        : y,
    'Pred_Train'    : y_train.round(4),
    'Pred_LOO'      : y_loo.round(4),
    'Residual'      : (y_loo - y).round(4),
    'Abs_Error'     : np.abs(y_loo - y).round(4)
})
print(df_res.to_string(index=False))
print(f'\nMean Abs Error: {np.abs(y_loo-y).mean():.4f}')